<a href="https://colab.research.google.com/github/teungkumeutuwah/Eksperimen_SML_muhammad-aiyub/blob/main/Streamlit_submission_BFGAI_Muhammad_Aiyub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📦 BAGIAN 2: STREAMLIT NOTEBOOK (Kriteria 3)


📋 Cell Streamlit 1: Install Dependencies


In [6]:
# ============================================
# STREAMLIT CELL 1: INSTALL STREAMLIT DEPENDENCIES
# ============================================
!pip install -q pyngrok streamlit streamlit-drawable-canvas==0.8.0
!pip list | grep streamlit-drawable-canvas

from pyngrok import ngrok
import subprocess
import time

print("✅ Streamlit dependencies ready!")

streamlit-drawable-canvas             0.8.0
✅ Streamlit dependencies ready!


📋 Cell Streamlit 2: Write logic.py


In [7]:
# ============================================
# STREAMLIT CELL 2: WRITE logic.py (BACKEND)
# ============================================
%%writefile logic.py
"""
===================================================================
LOGIC.PY - Backend Logic untuk Streamlit Interface
===================================================================
Berisi fungsi-fungsi yang dipanggil oleh app.py
"""

import torch
import gc
import numpy as np
from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionInpaintPipeline,
    StableDiffusionImg2ImgPipeline,
    EulerAncestralDiscreteScheduler,
    DPMSolverMultistepScheduler,
    DDIMScheduler,
)
from PIL import Image, ImageDraw, ImageFilter

print("✅ logic.py loaded")


def load_models_cached():
    """Load SD v1.5 + Inpainting models ke memory."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"📱 Loading models to {device}...")

    # Text-to-Image Pipeline
    pipe_txt2img = StableDiffusionPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        safety_checker=None,
    ).to(device)
    pipe_txt2img.enable_attention_slicing()

    # Inpainting Pipeline
    pipe_inpaint = StableDiffusionInpaintPipeline.from_pretrained(
        "runwayml/stable-diffusion-inpainting",
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        safety_checker=None,
    ).to(device)
    pipe_inpaint.enable_attention_slicing()

    # Img2Img untuk Two-Stage (Advanced) – opsional
    pipe_img2img = StableDiffusionImg2ImgPipeline(**pipe_txt2img.components).to(device)
    pipe_img2img.enable_attention_slicing()

    print("✅ All models loaded!")
    return pipe_txt2img, pipe_inpaint, pipe_img2img


def flush_memory():
    """Clear GPU RAM."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("🧹 Memory Flushed!")


def set_scheduler(pipe, scheduler_name):
    """
    Ganti scheduler tanpa reload model.
    Perbaikan: mendukung berbagai variasi penulisan dan fallback ke Euler A.
    """
    config = pipe.scheduler.config
    name = scheduler_name.lower().strip()

    # Debug (bisa dihapus setelah berhasil)
    print(f"🔧 Setting scheduler to: '{name}'")

    if name in ("euler a", "euler-a", "euler_ancestral", "euler_a"):
        pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(config)
        print("   ✅ Euler A scheduler applied")
    elif name in ("dpm++", "dpm++", "dpmsolver_multistep", "dpm"):
        pipe.scheduler = DPMSolverMultistepScheduler.from_config(config)
        print("   ✅ DPM++ scheduler applied")
    elif name == "ddim":
        pipe.scheduler = DDIMScheduler.from_config(config)
        print("   ✅ DDIM scheduler applied")
    else:
        # Fallback: jika tidak dikenali, pakai Euler A
        print(f"   ⚠️ Scheduler '{scheduler_name}' tidak dikenali, menggunakan Euler A.")
        pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(config)

    return pipe


def set_seed(seed):
    """Set random seed dengan device otomatis."""
    torch.manual_seed(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    return torch.Generator(device=device).manual_seed(seed)


def generate_image(pipe, prompt, neg_prompt, seed, steps, cfg, num_images=1):
    """Generate gambar (Text-to-Image)."""
    generator = set_seed(int(seed))

    result = pipe(
        prompt=[prompt] * num_images,
        negative_prompt=[neg_prompt] * num_images,
        num_inference_steps=int(steps),
        guidance_scale=float(cfg),
        generator=generator,
        width=512,
        height=768,  # Portrait
    )

    return result.images


def run_inpainting(pipe, image, mask, prompt, strength=0.8):
    """Jalankan inpainting."""
    if image.mode != "RGB":
        image = image.convert("RGB")
    if mask.mode != "L":
        mask = mask.convert("L")
    if mask.size != image.size:
        mask = mask.resize(image.size, resample=Image.NEAREST)

    result = pipe(
        prompt=prompt,
        image=image,
        mask_image=mask,
        strength=float(strength),
        num_inference_steps=50,
        guidance_scale=7.5,
    )

    return result.images[0]


def prepare_outpainting(image, expand_pixels=128):
    """Siapkan canvas untuk outpainting (zoom out)."""
    w, h = image.size
    new_w, new_h = w + expand_pixels * 2, h + expand_pixels * 2

    bg = image.resize((new_w, new_h), resample=Image.BICUBIC)
    bg = bg.filter(ImageFilter.GaussianBlur(radius=50))

    canvas = bg.copy()
    paste_x, paste_y = (new_w - w) // 2, (new_h - h) // 2
    canvas.paste(image, (paste_x, paste_y))

    mask = Image.new("L", (new_w, new_h), 255)
    inner_box = Image.new("L", (w, h), 0)
    mask.paste(inner_box, (paste_x, paste_y))

    return canvas, mask


print("✅ logic.py - All functions ready!")

Overwriting logic.py


📋 Cell Streamlit 3: Write app.py (Complete)


In [8]:
# ============================================
# STREAMLIT CELL 3: WRITE app.py (INTERFACE)
# ============================================
%%writefile app.py
"""
===================================================================
APP.PY - Streamlit Interface untuk Image Generation Studio
===================================================================
"""

import streamlit as st
import numpy as np
from PIL import Image
from streamlit_drawable_canvas import st_canvas
import logic

# ============================================
# PAGE CONFIG
# ============================================
st.set_page_config(
    page_title="🎨 StudioAI: Generative Image Suite",
    layout="wide",
    page_icon="🚀"
)

# ============================================
# LOAD MODELS (Cached)
# ============================================
@st.cache_resource
def get_models():
    return logic.load_models_cached()

try:
    pipe_txt2img, pipe_inpaint, pipe_img2img = get_models()
except Exception as e:
    st.error(f"❌ Error loading models: {e}")
    st.stop()

# ============================================
# HEADER
# ============================================
st.title("🎨 StudioAI: Creating Amazing Paint with Stable Diffusion")
st.markdown("*Generative Image Suite - End to End Solution*")

# ============================================
# SIDEBAR: PARAMETERS
# ============================================
with st.sidebar:
    st.header("⚙️ Parameters")

    # --- BASIC ---
    st.subheader("Basic")
    steps = st.slider("Quality Steps", 15, 50, 30)
    cfg = st.slider("Creativity (CFG)", 1.0, 20.0, 7.5)
    seed = st.number_input("Seed Control", value=42)

    st.divider()

    # --- SKILLED ---
    st.subheader("✅ Advanced Options")
    scheduler_name = st.selectbox(
        "Scheduler",
        ["Euler A", "DPM++", "DDIM"]
    )
    num_images = st.slider("Batch Size (Jumlah Gambar)", 1, 4, 1)

    st.divider()

    # --- MEMORY ---
    if st.button("🧹 Clear Memory"):
        logic.flush_memory()
        st.toast("Memory Cleared!")

# ============================================
# TABS
# ============================================
tab_gen, tab_edit = st.tabs(["🖼️ GENERATE", "✏️ EDIT (Inpaint/Outpaint)"])

# ============================================
# TAB 1: GENERATE
# ============================================
with tab_gen:
    st.header("Generate Image from Text")

    col_input, col_output = st.columns([1, 1], gap="large")

    with col_input:
        st.subheader("📝 Input Blueprint")

        with st.form(key="gen_form"):
            prompt = st.text_area(
                "Prompt",
                value="""(masterpiece, best quality:1.2),
(cute astronaut:1.3) in white space suit, NASA suit,
(standing pose:1.2), (full body visible:1.3),
(head with helmet clearly visible:1.4), (face through visor:1.3),
centered composition, on moon surface, gray rocky ground, craters,
outer space background, stars, Earth planet in distance, blue and white earth,
cinematic lighting, dramatic shadows, sunlight from right side,
highly detailed, realistic, professional photography, 4k, sharp focus""",
                height=180
            )

            neg_prompt = st.text_input(
                "Negative Prompt",
                value="(cropped:1.6), (head out of frame:1.6), cartoon, blurry, low quality, bad anatomy"
            )

            submitted = st.form_submit_button("🚀 Initialize Generation")

        if submitted:
            with st.spinner("Processing..."):
                logic.flush_memory()

                # Set scheduler
                pipe_txt2img = logic.set_scheduler(pipe_txt2img, scheduler_name)

                # Generate
                generated = logic.generate_image(
                    pipe_txt2img, prompt, neg_prompt,
                    seed, steps, cfg, num_images
                )

                st.session_state['generated_images'] = generated

                if generated:
                    st.session_state['current_image'] = generated[0]

                st.rerun()

    with col_output:
        st.subheader("🖼️ Visual Output")

        if 'generated_images' in st.session_state:
            imgs = st.session_state['generated_images']

            if len(imgs) > 1:
                # Grid display untuk batch
                cols = st.columns(2)
                for idx, img in enumerate(imgs):
                    with cols[idx % 2]:
                        st.image(img, caption=f"Image {idx+1}", use_container_width=True)
            else:
                st.image(imgs[0], caption="Generated Result", use_container_width=True)

            if st.button("📌 Select for Editing"):
                st.session_state['current_image'] = imgs[0]
                st.success("✅ Image selected for editing!")
        else:
            st.info("💡 Enter prompt and click Generate.")

# ============================================
# TAB 2: EDIT (INPAINTING & OUTPAINTING)
# ============================================
with tab_edit:
    st.header("Edit Image (Inpainting / Outpainting)")

    # Validasi gambar
    if 'current_image' not in st.session_state:
        st.warning("⚠️ Please generate an image first in the GENERATE tab.")
        source_img = None
    else:
        source_img = st.session_state['current_image']
        st.session_state['source_image'] = source_img

    # Mode selection
    edit_mode = st.radio(
        "Select Mode:",
        ["Inpainting (Edit Object)", "Outpainting (Zoom Out)"],
        horizontal=True
    )

    # ============================================
    # MODE: INPAINTING
    # ============================================
    if edit_mode == "Inpainting (Edit Object)":
        col_canvas, col_settings = st.columns([1, 1], gap="large")

        with col_canvas:
            st.subheader("🎨 Draw Mask")
            st.caption("Draw on the area you want to change.")

            if source_img:
                W, H = source_img.size
                canvas_result = st_canvas(
                    fill_color="rgba(255, 255, 255, 1.0)",
                    stroke_width=20,
                    stroke_color="#FFFFFF",
                    background_image=source_img,
                    update_streamlit=True,
                    height=H,
                    width=W,
                    drawing_mode="freedraw",
                    key=f"inpaint_canvas_{st.session_state.get('canvas_key', 0)}"
                )

        with col_settings:
            st.subheader("⚙️ Settings")

            with st.form(key="inpaint_form"):
                edit_prompt = st.text_area(
                    "New Prompt (What to add?)",
                    value="broken satellite, crashed space probe, metallic debris on moon surface",
                    height=100
                )
                strength = st.slider("Strength", 0.5, 1.0, 0.8)
                submit_inpaint = st.form_submit_button("✏️ Execute Inpainting")

            if submit_inpaint:
                if canvas_result.image_data is not None and np.max(canvas_result.image_data) > 0:
                    with st.spinner("Processing Inpainting..."):
                        logic.flush_memory()

                        # Extract mask from canvas alpha channel
                        mask_data = canvas_result.image_data[:, :, 3]
                        mask_image = Image.fromarray(mask_data).convert("L")
                        mask_image = mask_image.point(lambda x: 255 if x > 0 else 0)

                        if mask_image.size != source_img.size:
                            mask_image = mask_image.resize(source_img.size)

                        try:
                            result = logic.run_inpainting(
                                pipe_inpaint, source_img, mask_image,
                                edit_prompt, strength
                            )

                            st.session_state['current_image'] = result
                            st.session_state['canvas_key'] = int(st.session_state.get('canvas_key', 0)) + 1
                            st.success("✅ Inpainting Successful!")
                            st.rerun()

                        except Exception as e:
                            st.error(f"❌ Error: {e}")
                else:
                    st.warning("⚠️ Please draw on the image first.")

    # ============================================
    # MODE: OUTPAINTING
    # ============================================
    elif edit_mode == "Outpainting (Zoom Out)":
        col_orig, col_out = st.columns([1, 1], gap="large")

        with col_orig:
            st.subheader("📐 Original")
            if source_img:
                st.image(source_img, use_container_width=True)

        with col_out:
            st.subheader("🔍 Expanded (Zoom Out)")
            out_prompt = st.text_input(
                "Outpaint Prompt",
                value="continuation of the scene, seamless extension, consistent style"
            )

            if st.button("🔍 Zoom Out"):
                if source_img:
                    with st.spinner("Expanding Canvas..."):
                        logic.flush_memory()

                        try:
                            canvas_ready, mask_ready = logic.prepare_outpainting(
                                source_img, expand_pixels=128
                            )

                            final_out = logic.run_inpainting(
                                pipe_inpaint, canvas_ready, mask_ready,
                                out_prompt, 0.8
                            )

                            st.session_state['current_image'] = final_out
                            st.success("✅ Outpainting Complete!")
                            st.rerun()

                        except Exception as e:
                            st.error(f"❌ Error: {e}")

# ============================================
# FOOTER
# ============================================
st.divider()
st.caption("Built with ❤️ using Stable Diffusion + Streamlit | Muhammad Aiyub | Proyek Image Generation")

Overwriting app.py


📋 Cell Streamlit 4: Run dengan Ngrok


In [9]:
# ============================================
# STREAMLIT CELL 4: RUN STREAMLIT VIA NGROK
# ============================================

# ============================================
# CONFIGURATION
# ============================================
AUTH_TOKEN = "3GuDxBIcBNlbD1jD2Mzv5TKX6Nd_5yoT31Rs4QEJ3TA6HfFWm"  # GANTI DENGAN TOKEN ASLI ANDA
PORT = 8501

# Cek apakah Streamlit sudah jalan
result = subprocess.run(["lsof", "-i", f":{PORT}"], capture_output=True, text=True)
print(f"Port {PORT} status:", "USED" if result.stdout else "FREE")

# ============================================
# STEP 1: Cleanup tunnel lama
# ============================================
print("\n🔧 Cleaning up old tunnels...")
try:
    for tunnel in ngrok.get_tunnels():
        ngrok.disconnect(tunnel.public_url)
        print(f"  ❌ Disconnected: {tunnel.public_url}")
except Exception as e:
    print(f"  ℹ️ {e}")

# Kill proses lama
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
subprocess.run(["pkill", "-f", "ngrok"], capture_output=True)
print("  ✅ Old processes killed")

time.sleep(3)

# ============================================
# STEP 2: Set Auth Token
# ============================================
print("\n🔑 Setting Ngrok auth token...")
ngrok.set_auth_token(AUTH_TOKEN)
print("  ✅ Token configured!")

# ============================================
# STEP 3: Start Streamlit
# ============================================
print(f"\n🚀 Starting Streamlit on port {PORT}...")

log_file = open("/tmp/streamlit.log", "w")

proc = subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", str(PORT),
    "--server.headless", "true",
    "--server.address", "0.0.0.0"
], stdout=log_file, stderr=log_file)

time.sleep(8)  # Tunggu load model

# Verifikasi
check = subprocess.run(["lsof", "-i", f":{PORT}"], capture_output=True, text=True)
if check.stdout:
    print("  ✅ Streamlit confirmed running!")
else:
    print("  ❌ Streamlit failed to start! Check log:")
    !cat /tmp/streamlit.log | head -20

# ============================================
# STEP 4: Connect Ngrok
# ============================================
print("\n🌐 Connecting Ngrok tunnel...")
tunnel = ngrok.connect(PORT)
url = tunnel.public_url.replace("https://", "https://")

print(f"\n{'='*60}")
print(f"✅ APPLICATION READY!")
print(f"{'='*60}")
print(f"\n🔗 CLICK THIS URL TO OPEN IN BROWSER:\n")
print(f"   👉 {url}\n")
print(f"{'='*60}")

Port 8501 status: FREE

🔧 Cleaning up old tunnels...
  ❌ Disconnected: https://remedial-scrawny-unworn.ngrok-free.dev
  ✅ Old processes killed

🔑 Setting Ngrok auth token...
  ✅ Token configured!

🚀 Starting Streamlit on port 8501...
  ❌ Streamlit failed to start! Check log:



🌐 Connecting Ngrok tunnel...

✅ APPLICATION READY!

🔗 CLICK THIS URL TO OPEN IN BROWSER:

   👉 https://remedial-scrawny-unworn.ngrok-free.dev



In [10]:
# ============================================
# STREAMLIT CELL 5: GENERATE requirements.txt
# ============================================
%%writefile requirements.txt
# Core Deep Learning & Diffusion
torch>=2.0.0
torchvision>=0.15.0
diffusers>=0.21.0
transformers>=4.30.0
accelerate>=0.20.0
safetensors>=0.3.0
xformers>=0.0.20

# Computer Vision & Image Processing
opencv-python-headless>=4.7.0
Pillow>=9.0.0
matplotlib>=3.7.0
numpy>=1.24.0
scipy>=1.10.0

# Segmentation (untuk auto-mask di pipeline)
segment-anything>=1.0

# Streamlit Interface
streamlit>=1.28.0
streamlit-drawable-canvas>=0.8.0
pyngrok>=5.0.0

# Utilities
requests>=2.28.0

print("\n✅ requirements.txt berhasil dibuat!")
print("="*50)
!cat requirements.txt

Overwriting requirements.txt
